In [17]:
import time
import chess
import chess.svg
from IPython.display import display, clear_output, SVG

# ==========================================
# 1. MINIMAX ENGINE WITH EVALUATION
# ==========================================
PIECE_VALUES = {
    chess.PAWN: 100, chess.KNIGHT: 320, chess.BISHOP: 330,
    chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 20000
}

PAWN_PST = [
     0,  0,  0,  0,  0,  0,  0,  0,
    50, 50, 50, 50, 50, 50, 50, 50,
    10, 10, 20, 30, 30, 20, 10, 10,
     5,  5, 10, 25, 25, 10,  5,  5,
     0,  0,  0, 20, 20,  0,  0,  0,
     5, -5,-10,  0,  0,-10, -5,  5,
     5, 10, 10,-20,-20, 10, 10,  5,
     0,  0,  0,  0,  0,  0,  0,  0
]
REV_PAWN_PST=PAWN_PST
REV_PAWN_PST.reverse()

turn_name = None

def evaluate_board(board: chess.Board) -> int:
    global turn_name
    global REV_PAWN_PST
    """Static evaluation function returning White's perspective score."""
    if board.is_checkmate():
        return -99999 if board.turn == chess.WHITE else 99999
    if board.is_stalemate() or board.is_insufficient_material():
        return 0

    score = 0
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is None:
            continue

        val = PIECE_VALUES[piece.piece_type]
        if piece.piece_type == chess.PAWN:
            idx = square if piece.color == chess.WHITE else chess.square_mirror(square)
            if turn_name == "Black (AI)":
                val += REV_PAWN_PST[idx]
            else:
                val += PAWN_PST[idx]

        if piece.color == chess.WHITE:
            score += val
        else:
            score -= val
    return score

def minimax(board: chess.Board, depth: int, is_max: bool) -> int:
    """Minimax recursion."""
    if depth == 0 or board.is_game_over():
        return evaluate_board(board)

    if is_max:
        best = float('-inf')
        for move in board.legal_moves:
            board.push(move)
            best = max(best, minimax(board, depth - 1, False))
            board.pop()
        return best
    else:
        best = float('inf')
        for move in board.legal_moves:
            board.push(move)
            best = min(best, minimax(board, depth - 1, True))
            board.pop()
        return best

def get_ai_move(board: chess.Board, depth: int = 3) -> chess.Move:
    """Selects the optimal move using Minimax."""
    best_move = None
    is_max = (board.turn == chess.WHITE)
    best_score = float('-inf') if is_max else float('inf')

    for move in board.legal_moves:
        board.push(move)
        score = minimax(board, depth - 1, not is_max)
        board.pop()

        if is_max and score > best_score:
            best_score = score
            best_move = move
        elif not is_max and score < best_score:
            best_score = score
            best_move = move

    return best_move


# ==========================================
# 2. NOTEBOOK RENDERING & GAME LOOP
# ==========================================
def render_notebook_board(board: chess.Board, last_move=None):
    """Refreshes the Jupyter cell output with an SVG chessboard."""
    clear_output(wait=True)
    
    # Generate interactive SVG board
    board_svg = chess.svg.board(
        board,
        size=400,
        lastmove=last_move,
        check=board.king(board.turn) if board.is_check() else None
    )
    display(SVG(board_svg))


def play_chess_game(mode="human_vs_ai", user_color=chess.WHITE, depth=3, delay=0.5):
    """
    Main loop for playing in Jupyter.
    
    :param mode: "human_vs_ai" or "ai_vs_ai"
    :param user_color: chess.WHITE or chess.BLACK (for human_vs_ai mode)
    :param depth: Minimax evaluation depth
    :param delay: Pause duration between turns in seconds (AI vs AI)
    """
    global turn_name
    
    board = chess.Board()
    last_move = None

    while not board.is_game_over():
        print(board)
        #render_notebook_board(board, last_move=last_move)
        
        is_human_turn = (mode == "human_vs_ai") and (board.turn == user_color)

        if is_human_turn:
            print("Your turn! (Enter move in UCI format, e.g., 'e2e4' or 'e7e8q')")
            valid_move = False
            while not valid_move:
                try:
                    move_str = input("Move: ").strip()
                    move = chess.Move.from_uci(move_str)
                    if move in board.legal_moves:
                        board.push(move)
                        last_move = move
                        valid_move = True
                    else:
                        print("Illegal move! Legal moves are:", [m.uci() for m in board.legal_moves])
                except ValueError:
                    print("Invalid UCI string! Format: e2e4, g1f3, etc.")
        else:
            turn_name = "White (AI)" if board.turn == chess.WHITE else "Black (AI)"
            print(f"Thinking for {turn_name} at depth {depth}...")
            
            ai_move = get_ai_move(board, depth=depth)
            if ai_move:
                board.push(ai_move)
                last_move = ai_move
            
            time.sleep(delay)

    # Render final game state
    render_notebook_board(board, last_move=last_move)
    print("Game Over!")
    print(f"Result: {board.result()}")

In [18]:
play_chess_game()

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
Your turn! (Enter move in UCI format, e.g., 'e2e4' or 'e7e8q')


Move:  e2e4


r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R
Thinking for Black (AI) at depth 3...
r n b q k b n r
p p p p . p p p
. . . . . . . .
. . . . p . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R
Your turn! (Enter move in UCI format, e.g., 'e2e4' or 'e7e8q')


Move:  g7f3


Illegal move! Legal moves are: ['g1h3', 'g1f3', 'g1e2', 'f1a6', 'f1b5', 'f1c4', 'f1d3', 'f1e2', 'e1e2', 'd1h5', 'd1g4', 'd1f3', 'd1e2', 'b1c3', 'b1a3', 'h2h3', 'g2g3', 'f2f3', 'd2d3', 'c2c3', 'b2b3', 'a2a3', 'h2h4', 'g2g4', 'f2f4', 'd2d4', 'c2c4', 'b2b4', 'a2a4']


...

Move:  surrender


Invalid UCI string! Format: e2e4, g1f3, etc.


Move:  exit


Invalid UCI string! Format: e2e4, g1f3, etc.


KeyboardInterrupt: Interrupted by user

In [ ]:
                        last_move = move
